# LAi Training on Google Colab
Train the LAi bilingual (HU/EN) model on a free T4 GPU.

**Steps:**
1. Clone repo & install dependencies
2. Prepare training data (chat-formatted HU/EN)
3. Build BPE vocabulary
4. Train model on GPU
5. Download trained model

**Runtime → Change runtime type → T4 GPU**

In [ ]:
# 0. Check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
import torch
print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 1. Clone repo & install deps
!git clone https://github.com/ML6200/LAi2.git
%cd LAi2
!pip install -q psutil datasets tqdm

In [ ]:
# 2. Prepare training data
#
# IMPORTANT: Use 'small' or larger for good results.
# 'micro'/'tiny' datasets are too small and the model will memorize them.
#
# Sizes:
#   small:  10K translations + 10K conversations + 8K HU + 8K stories = ~36K examples
#   medium: 25K translations + 25K conversations + 20K HU + 20K stories = ~90K examples
#   large:  50K + 50K + 40K + 40K = ~180K examples

DATASET_SIZE = "small"  # @param ["small", "medium", "large"]

!python training/data.py --output data/train.txt --size {DATASET_SIZE}

# Verify dataset
!echo "---"
!wc -l data/train.txt
!sort -u data/train.txt | wc -l | xargs -I{} echo "{} unique examples"

In [ ]:
# 3. Build BPE vocabulary
VOCAB_SIZE = 16000  # @param {type:"integer"}

!python training/build_vocab.py \
    --data data/train.txt \
    --vocab_size {VOCAB_SIZE} \
    --output data/vocab.bin

In [ ]:
# 4. Train!
#
# Model configs:
#   tiny:  ~20M params, 384 dim, 8 layers  (~15 min on T4 with small dataset)
#   mini:  ~80M params, 512 dim, 12 layers (~45 min on T4 with small dataset)
#
# IMPORTANT: Don't overtrain!
#   - With 'small' dataset (36K examples): use 5-8 epochs
#   - With 'medium' dataset (90K examples): use 3-5 epochs
#   - Target final loss: 1.5-2.5 (NOT below 1.0 — that means memorization)
#   - If loss drops below 0.5, the model is overfitting

MODEL_CONFIG = "tiny"  # @param ["tiny", "mini"]
EPOCHS = 5  # @param {type:"integer"}
BATCH_SIZE = 64  # @param {type:"integer"}
GRAD_ACCUM = 4  # @param {type:"integer"}
LEARNING_RATE = 3e-4  # @param {type:"number"}

!python training/train.py \
    --config {MODEL_CONFIG} \
    --data data/train.txt \
    --vocab data/vocab.bin \
    --output models/lai-{MODEL_CONFIG}.bin \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --grad_accum {GRAD_ACCUM} \
    --lr {LEARNING_RATE} \
    --device cuda

In [ ]:
# 5. Quick sanity check: verify model file
import os, struct

model_path = f'models/lai-{MODEL_CONFIG}.bin'
size_mb = os.path.getsize(model_path) / 1e6

with open(model_path, 'rb') as f:
    magic = f.read(4)
    version = struct.unpack('I', f.read(4))[0]
    dim = struct.unpack('i', f.read(4))[0]
    hidden = struct.unpack('i', f.read(4))[0]
    layers = struct.unpack('i', f.read(4))[0]
    heads = struct.unpack('i', f.read(4))[0]

print(f"Model: {model_path}")
print(f"  Size: {size_mb:.1f} MB")
print(f"  Magic: {magic}")
print(f"  Config: {dim}d, {layers}L, {heads}H")
print(f"\nReady to download!")

In [ ]:
# 6. Download the trained model
from google.colab import files

print(f"Downloading {model_path} ({size_mb:.1f} MB)...")
files.download(model_path)

## After downloading

Copy the model to your LAi project and test:

```bash
# On your Mac
mv ~/Downloads/lai-tiny.bin models/
make clean && make release
./lai -m models/lai-tiny.bin -p "Szia, hogy vagy?"
```

### Understanding loss values
| Final loss | Meaning |
|---|---|
| > 4.0 | Model hasn't learned much — need more data or epochs |
| 2.0-3.0 | Good generalization — model understands patterns |
| 1.0-2.0 | Strong — coherent output with some creativity |
| < 0.5 | **Overfitting** — model memorized training data, reduce epochs |

### Recommended configs
| Goal | Dataset | Model | Epochs |
|---|---|---|---|
| Quick test | small | tiny | 5 |
| Good quality | medium | tiny | 3-5 |
| Best quality | large | mini | 3-5 |